## Looking At the Schema and Rows of Bronze Tables before Any Transformation

In [0]:
tables = [
    "bts_flight_data.bronze.flights_raw",
    "bts_flight_data.bronze.airports_lookup",
    "bts_flight_data.bronze.carriers_lookup",
]

for table in tables:
    print(f"\n{'='*60}")
    print(f"TABLE: {table}")
    print('='*60)

    print("\n--- Schema ---")
    spark.sql(f"DESCRIBE {table}").show(50, truncate=False)

    print("\n--- Row count ---")
    count = spark.sql(f"SELECT COUNT(*) as cnt FROM {table}").collect()[0]["cnt"]
    print(f"Rows: {count:,}")

    print("\n--- Sample rows ---")
    spark.sql(f"SELECT * FROM {table} LIMIT 3").show(truncate=False, vertical=True)

## Viewing Each Silver Layer Staging Table Separately

In [0]:
%sql
SELECT * FROM bts_flight_data.silver.stg_flights;

In [0]:
%sql
SELECT COUNT(*) FROM bts_flight_data.silver.stg_flights_completed;

In [0]:
%sql
SELECT COUNT(*) FROM bts_flight_data.silver.stg_flights_cancelled_diverted;

In [0]:
%sql
SELECT * FROM bts_flight_data.silver.stg_airports LIMIT 5;

In [0]:
%sql
SELECT * FROM bts_flight_data.silver.stg_carriers LIMIT 5;

## Viewing Snapshot Tables for Reviewing SCD Type 2

In [0]:
%sql
SELECT airport_code, airport_name, city, dbt_valid_from, dbt_valid_to
FROM bts_flight_data.silver.airports_snapshot
WHERE airport_code = 'ORD'
ORDER BY dbt_valid_from;

In [0]:
%sql
SELECT airport_code, COUNT(*) as version_count
FROM bts_flight_data.silver.airports_snapshot
GROUP BY airport_code
HAVING COUNT(*) > 1;

## Reviewing Intermediate Flights Enriched Table

In [0]:
%sql
SELECT tail_number, fl_date, dep_time, arr_delay, prev_flight_arr_delay
FROM bts_flight_data.silver.int_flights_enriched
WHERE tail_number = 'N104NN'
ORDER BY fl_date, dep_time
LIMIT 10;

In [0]:
%sql
SELECT COUNT(*) as total,
       SUM(CASE WHEN arr_delay_15plus THEN 1 ELSE 0 END) as delayed_count
FROM bts_flight_data.silver.int_flights_enriched;

# Gold Layer — Mart Usage

## Look up a specific airport's current details

In [0]:
%sql
SELECT airport_code, airport_name, city, country, timezone
FROM bts_flight_data.gold.dim_airports
WHERE airport_code = 'ORD'
  AND is_current = true;

## All currently active carriers, alphabetically

In [0]:
%sql
SELECT carrier_code, carrier_name, country
FROM bts_flight_data.gold.dim_carriers
WHERE active = 'Y'
  AND is_current = true
ORDER BY carrier_name;

## Flights where the previous aircraft delay was severe (60+ min)

In [0]:
%sql
SELECT
    fl_date,
    flight_number,
    prev_flight_arr_delay,
    dep_delay,
    arr_delay,
    arr_delay_15plus
FROM bts_flight_data.gold.fct_flights
WHERE prev_flight_arr_delay >= 60
ORDER BY prev_flight_arr_delay DESC
LIMIT 20;

## All severely delayed flights out of a specific origin airport

In [0]:
%sql
SELECT
    f.fl_date,
    f.flight_number,
    dc.carrier_name,
    da.airport_name AS origin_airport,
    f.dep_delay,
    f.arr_delay
FROM bts_flight_data.gold.fct_flights f
JOIN bts_flight_data.gold.dim_carriers dc ON f.carrier_sk = dc.carrier_sk
JOIN bts_flight_data.gold.dim_airports da ON f.origin_airport_sk = da.airport_sk
WHERE da.airport_code = 'ORD'
  AND f.arr_delay >= 60
ORDER BY f.fl_date;

## Worst-performing carriers by % delayed, most recent month available

In [0]:
%sql
SELECT carrier_name, flight_month, total_flights, pct_delayed
FROM bts_flight_data.gold.agg_carrier_monthly_performance
WHERE flight_year = 2025
ORDER BY flight_month DESC, pct_delayed DESC
LIMIT 10;

## Carrier delay % trend across the year

In [0]:
%sql
SELECT carrier_code, flight_month, pct_delayed
FROM bts_flight_data.gold.agg_carrier_monthly_performance
WHERE carrier_code IN ('AA', 'DL', 'UA', 'WN')
ORDER BY flight_month

## Most congested routes by average arrival delay

In [0]:
%sql
SELECT origin, dest, total_flights, avg_arr_delay, avg_congestion_score
FROM bts_flight_data.gold.agg_route_delay_trends
WHERE total_flights >= 100
ORDER BY avg_arr_delay DESC
LIMIT 15;